# HydroKG end-to-end demo

Runs the full offline audit, real-time audit, and all three enhancement mechanisms against synthetic basin data using the in-memory graph backend -- no CAMELS data, no Neo4j server, and no PyTorch required. Swap `build_graph_store("memory")` for `build_graph_store("neo4j", uri=..., user=..., password=...)` to run against production Neo4j; swap the synthetic basins for `hydrokg.adapters.load_predictions_pickle(...)` output to run against a real submodule evaluation run.

In [ ]:
import pandas as pd

from hydrokg.audit.offline_auditor import OfflineAuditor
from hydrokg.audit.realtime_auditor import RealtimeAuditor
from hydrokg.data.synthetic import make_synthetic_basin
from hydrokg.enhancement.curriculum import ViolationCurriculumSampler
from hydrokg.enhancement.graph_analogy_correction import GraphAnalogyCorrector
from hydrokg.enhancement.violation_embeddings import build_embedding_matrix
from hydrokg.evaluation.skill_trust_analysis import summarize_skill_trust
from hydrokg.graph.factory import build_graph_store
from hydrokg.viz.skill_trust_plots import plot_skill_trust_scatter
from hydrokg.viz.violation_summary_plots import plot_rule_violation_counts, plot_stratified_burden
from hydrokg.evaluation.stratification import stratified_violation_summary

## 1. Build synthetic basins with known, controlled violations

In [ ]:
basins = {
    "DEMO0001": make_synthetic_basin("DEMO0001", seed=1),
    "DEMO0002": make_synthetic_basin("DEMO0002", seed=2, inject_negative_flow=True),
    "DEMO0003": make_synthetic_basin("DEMO0003", seed=3, inject_extreme_ratio=True),
    "DEMO0004": make_synthetic_basin("DEMO0004", seed=4, inject_zero_collapse=True),
    "DEMO0005": make_synthetic_basin("DEMO0005", seed=5, inject_peak_lag_days=4),
    "DEMO0006": make_synthetic_basin("DEMO0006", seed=6, inject_mass_balance_violation=True),
    "DEMO0007": make_synthetic_basin("DEMO0007", seed=7),
}
stratification = pd.DataFrame({
    "aridity_class": ["humid", "sub_humid", "semi_arid", "humid", "sub_humid", "arid", "humid"],
    "landcover_class": ["forest", "forest", "shrubland", "cropland", "forest", "shrubland", "forest"],
}, index=list(basins.keys()))
basins["DEMO0002"].head()

## 2. Offline audit: apply R0-R6, compute violation burden (Eq. 3), compare against KGE

In [ ]:
graph = build_graph_store("memory")
auditor = OfflineAuditor(graph)
results = auditor.audit_all(basins, stratification)
results[["basin_id", "kge", "violation_burden", "dominant_class"]]

In [ ]:
summarize_skill_trust(results)

In [ ]:
plot_skill_trust_scatter(results);
plot_rule_violation_counts({row.basin_id: row.violation_counts for row in results.itertuples()});
plot_stratified_burden(stratified_violation_summary(results, by="aridity_class"), "aridity_class");

## 3. Real-time audit: stream one basin's predictions day by day

In [ ]:
rt_graph = build_graph_store("memory")
rt_auditor = RealtimeAuditor(rt_graph)
stream_df = make_synthetic_basin("STREAM", seed=42, inject_negative_flow=True, inject_peak_lag_days=3)
rt_auditor.register_basin("STREAM", aridity_class="humid", landcover_class="forest")
for ts, row in stream_df.iterrows():
    rt_auditor.ingest("STREAM", ts, row.qsim, row.qobs, row.p, "humid", "forest")
rt_auditor.flush_all()
rt_graph.get_violation_counts("STREAM")

## 4. The three graph-guided enhancement mechanisms

In [ ]:
# 1) Curriculum reweighting
sampler = ViolationCurriculumSampler(graph)
sampler.basin_weights(list(basins.keys()))

In [ ]:
# 2) Violation-history embeddings
build_embedding_matrix(graph, list(basins.keys()))

In [ ]:
# 3) Graph-analogy correction: fix DEMO0002's negative-flow violations using structurally
# similar, low-violation basins (DEMO0001/DEMO0007, same aridity+land-cover class)
corrector = GraphAnalogyCorrector(graph, basins)
negative_rows = basins["DEMO0002"][basins["DEMO0002"].qsim < 0]
for ts, row in negative_rows.head(5).iterrows():
    corrected, info = corrector.correct("DEMO0002", ts, row.qsim, "R0", "humid", "forest")
    print(f"{ts.date()}: raw={row.qsim:.3f} -> corrected={corrected:.3f} ({info['method']})")

## Next steps for a real run

- Replace the synthetic `basins` dict with `hydrokg.adapters.load_predictions_pickle(...)` pointed at a completed submodule evaluation run.
- Replace `stratification` with `hydrokg.data.basin_attributes.load_basin_stratification(db_path, basin_ids)`.
- Swap `build_graph_store("memory")` for `build_graph_store("neo4j", uri=..., user=..., password=...)` after `docker compose up -d`.
- For the full enhancement pipeline (fine-tuning + graph-analogy correction against a real checkpoint), see `hydrokg.enhancement.enhanced_training.EnhancedTrainingPipeline` -- flagged in its module docstring as not yet run end-to-end.